# CALM model benchmark on Colab

Scores candidate open-source models against the 44 reviewed questions in
`tests/fixtures/scope_eval.jsonl`, using the same deterministic checks as a local run.
No model judges another model.

**Why Colab:** the development laptop has 6 GB of VRAM, which caps it at roughly an 8B
model and even that spills to CPU. A Colab T4 has 16 GB, which comfortably fits 8-14B
and can just about hold a 20B at 4-bit.

**The benchmark runs here, inside Colab, not over a tunnel.** `calm_core` imports only
the standard library, so there is nothing to `pip install` -- and running it here keeps
the latency column meaningful. Scoring over a tunnel from the laptop would measure the
tunnel. There is an optional tunnel cell at the bottom for *interactive* use from Unity.

**Privacy note:** the 44 questions are a fixed reviewed fixture. No learner data of any
kind leaves any machine in this notebook.

**Before running:** Runtime -> Change runtime type -> T4 GPU.

## 1. Confirm a GPU is actually attached

In [ ]:
# Do not skip this. Colab will happily give you a CPU-only runtime, and a 9B model
# on CPU turns a 20-minute sweep into an overnight one -- it does not fail, it crawls.
import subprocess

probe = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                        '--format=csv,noheader'],
                       capture_output=True, text=True)
if probe.returncode != 0:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.')
print(probe.stdout.strip())

## 2. Install Ollama and start it

In [ ]:
import os, subprocess, time, urllib.request

# One model resident at a time. Ollama keeps a model warm for 5 minutes by default,
# so a sweep that steps 9B -> 14B -> 20B would try to hold two at once and fall back
# to CPU (or die) partway through a run that looks like it is working.
os.environ['OLLAMA_MAX_LOADED_MODELS'] = '1'
os.environ['OLLAMA_KEEP_ALIVE'] = '60s'

subprocess.run('curl -fsSL https://ollama.com/install.sh | sh',
               shell=True, check=True)

server = subprocess.Popen(['ollama', 'serve'], env=os.environ.copy(),
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(90):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=2)
        print('ollama is up')
        break
    except Exception:
        time.sleep(1)
else:
    raise SystemExit('ollama did not come up')

## 3. Upload the bundle

On the laptop, run `python scripts/make_colab_bundle.py` and upload the
`calm_bench_bundle.zip` it writes (about 1.6 MB).

In [ ]:
import zipfile, pathlib
from google.colab import files

PROJECT = pathlib.Path('/content/calm')

uploaded = files.upload()
name = next(iter(uploaded))
with zipfile.ZipFile(name) as bundle:
    bundle.extractall(PROJECT)

# Grounding is not optional: RAGChatService refuses to answer without the curated
# cards, so a bundle missing them would score every model as a deterministic fallback.
for needed in ['calm_core/rag_chat.py', 'corpus/protocol_cards.jsonl',
               'config/unity_scenario_crosswalk.v1.json',
               'tests/fixtures/scope_eval.jsonl']:
    assert (PROJECT / needed).exists(), f'bundle is missing {needed}'
print('bundle looks complete')

## 4. Choose and pull the models

Sizes are the 4-bit download. A T4 has 16 GB, so anything up to ~14B is comfortable
and `sailor2:20b` (12 GB) is tight but workable.

| Model | Size | Why it is on the list |
|---|---|---|
| `qwen2.5:3b` | 1.9 GB | The incumbent. Keep it as the control or the comparison means nothing. |
| `sailor2:8b` | 5.2 GB | Qwen2.5 continually pretrained on 500B SEA tokens; explicitly covers Tagalog, Cebuano, Ilocano, Waray. |
| `aisingapore/Llama-SEA-LION-v3.5-8B-R` | ~5 GB | AI Singapore, 11 SEA languages incl. Filipino. The `-R` is a reasoning variant. |
| `aisingapore/Gemma-SEA-LION-v3-9B-IT` | ~6 GB | Gemma-based SEA model; different base, so a useful second opinion. |
| `qwen2.5:14b` | ~9 GB | Pure size control. If this beats the SEA models, the problem was capacity, not language coverage. |
| `sailor2:20b` | 12 GB | The ceiling that fits a free T4. |

Start with the first four. Pulling all six is ~39 GB of download.

In [ ]:
MODELS = [
    'qwen2.5:3b',
    'sailor2:8b',
    'aisingapore/Llama-SEA-LION-v3.5-8B-R',
    'aisingapore/Gemma-SEA-LION-v3-9B-IT',
    # 'qwen2.5:14b',
    # 'sailor2:20b',
]

for model in MODELS:
    print('pulling', model, flush=True)
    subprocess.run(['ollama', 'pull', model], check=True)
print('\nall pulled')

## 5. Run the sweep

In [ ]:
env = os.environ.copy()
# A 14B answering in Filipino is slower per token than a 3B answering in English,
# and a timeout here would be scored as a deterministic fallback rather than as the
# model's own answer -- i.e. it would quietly flatter the model instead of failing.
env['CALM_OLLAMA_TIMEOUT_SECONDS'] = '180'
# Harmless here; the crosswalk's configured Unity path does not exist on Colab, so
# the live-drift check degrades to a warning on its own.
env['CALM_UNITY_DRIFT'] = 'warn'

report = PROJECT / 'MODEL_BENCHMARK_colab.md'
subprocess.run(
    ['python', 'scripts/benchmark_models.py',
     '--models', *MODELS,
     '--report', str(report),
     '--load-timeout', '600'],
    cwd=PROJECT, env=env, check=True)

print(report.read_text())

## 6. Download the report

In [ ]:
files.download(str(report))

## Optional: expose this Ollama to the laptop or the headset

Only for *interactive* testing -- asking KALMA questions from Unity against a big model.
Do not score through it: the numbers would include tunnel latency.

The quick tunnel is a public URL with no authentication. Anyone who has it can spend
your GPU time. It dies with the runtime; treat it as disposable and do not paste it
anywhere public.

In [ ]:
import re

subprocess.run(
    'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/'
    'cloudflared-linux-amd64 -O /usr/local/bin/cloudflared '
    '&& chmod +x /usr/local/bin/cloudflared', shell=True, check=True)

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:11434'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

public = None
for line in tunnel.stdout:
    found = re.search(r'https://[-\w.]+\.trycloudflare\.com', line)
    if found:
        public = found.group(0)
        break

print('CALM_OLLAMA_URL=' + (public or 'FAILED - check the cloudflared output'))
print('\nOn the laptop, in the calm-ai-assistant folder:')
print(f'  $env:CALM_OLLAMA_URL="{public}"')
print('  $env:CALM_OLLAMA_MODEL="sailor2:8b"')
print('  $env:CALM_UNITY_DRIFT="warn"; $env:CALM_WHISPER_DEVICE="cpu"')
print('  .\\venv\\Scripts\\python.exe -m uvicorn server:app --host 127.0.0.1 --port 8010')